In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Vivek_Vihar_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,364.0,192.0,223.0,131.0,205.0,263.0,100.0,51.0,78.0,215.0,358.0,304.0
1,2,359.0,264.0,102.0,127.0,178.0,185.0,120.0,61.0,70.0,242.0,353.0,295.0
2,3,334.0,260.0,103.0,170.0,232.0,163.0,103.0,50.0,76.0,153.0,404.0,286.0
3,4,384.0,323.0,132.0,216.0,243.0,228.0,63.0,52.0,72.0,188.0,431.0,171.0
4,5,379.0,249.0,127.0,168.0,316.0,276.0,80.0,50.0,89.0,160.0,406.0,143.0
5,6,370.0,183.0,112.0,159.0,261.0,194.0,59.0,51.0,77.0,132.0,381.0,218.0
6,7,378.0,159.0,172.0,157.0,345.0,204.0,47.0,NaN,86.0,108.0,420.0,250.0
7,8,NaN,174.0,NaN,174.0,222.0,258.0,52.0,53.0,93.0,204.0,412.0,325.0
8,9,395.0,159.0,115.0,221.0,NaN,168.0,NaN,NaN,98.0,198.0,382.0,185.0
9,10,304.0,331.0,138.0,199.0,154.0,162.0,135.0,56.0,85.0,114.0,NaN,259.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,364.000000,192.000000,223.0,131.000000,205.000000,263.000000,100.000000,51.000000,78.000000,215.0,358.00000,304.0
1,2,359.000000,264.000000,102.0,127.000000,178.000000,185.000000,120.000000,61.000000,70.000000,242.0,353.00000,295.0
2,3,334.000000,260.000000,103.0,170.000000,232.000000,163.000000,103.000000,50.000000,76.000000,153.0,404.00000,286.0
3,4,384.000000,323.000000,132.0,216.000000,243.000000,228.000000,63.000000,52.000000,72.000000,188.0,431.00000,171.0
4,5,379.000000,249.000000,127.0,168.000000,316.000000,276.000000,80.000000,50.000000,89.000000,160.0,406.00000,143.0
5,6,370.000000,183.000000,112.0,159.000000,261.000000,194.000000,59.000000,51.000000,77.000000,132.0,381.00000,218.0
6,7,378.000000,159.000000,172.0,157.000000,198.676471,204.000000,47.000000,60.529412,86.000000,108.0,420.00000,250.0
7,8,345.735294,174.000000,145.7,174.000000,222.000000,258.000000,52.000000,53.000000,93.000000,204.0,412.00000,325.0
8,9,395.000000,159.000000,115.0,221.000000,198.676471,168.000000,84.205882,60.529412,98.000000,198.0,382.00000,185.0
9,10,304.000000,331.000000,138.0,199.000000,154.000000,162.000000,135.000000,56.000000,85.000000,114.0,357.69697,259.0
